In [1]:
import re
from itertools import combinations

import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
from tqdm import tqdm

tqdm.pandas()

In [2]:
df = pd.read_parquet("data/slurm.parquet")
df

,Account,AdminComment,AllocCPUS,AllocNodes,AllocTRES,AssocID,AveCPU,AveCPUFreq,AveDiskRead,AveDiskWrite,...,TRESUsageOutMinNode,TRESUsageOutMinTask,TRESUsageOutTot,UID,User,UserCPU,WCKey,WCKeyID,WorkDir,Unnamed: 120
0,default,NaN,60,2,"billing=60,cpu=60,mem=1600G,node=2",1654,None,None,None,None,...,None,None,None,406601040.0,40311488,00:00.006,NaN,0.0,/mnt/scratch2/users/40311488,NaN
1,default,NaN,1,1,"cpu=1,mem=800G,node=1",1654,00:00:00,2.50M,79713.93M,1351.21M,...,"energy=node172,fs/disk=node172",fs/disk=0,"energy=294,fs/disk=1416846876",NaN,None,00:00.006,NaN,NaN,None,NaN
2,default,NaN,56,1,"billing=56,cpu=56,mem=900G,node=1",1883,None,None,None,None,...,None,None,None,406601291.0,40258564,00:00:00,NaN,0.0,/mnt/scratch2/users/40258564/C_I_Ionisation/n1...,NaN
3,default,NaN,56,1,"billing=56,cpu=56,mem=900G,node=1",1883,None,None,None,None,...,None,None,None,406601291.0,40258564,00:00:00,NaN,0.0,/mnt/scratch2/users/40258564/C_I_Ionisation/n1...,NaN
4,default,NaN,56,1,"billing=56,cpu=56,mem=900G,node=1",1883,None,None,None,None,...,None,None,None,406601291.0,40258564,00:00:00,NaN,0.0,/mnt/scratch2/users/40258564/C_I_Ionisation/n1...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10888397,default,NaN,128,1,"billing=128,cpu=128,mem=1031040M,node=1",1796,None,None,None,None,...,None,None,None,406601198.0,3057924,05:50:41,NaN,0.0,/users/3057924/msca-mip-ann-training/src,NaN
10888398,default,NaN,128,1,"cpu=128,mem=1031040M,node=1",1796,06:07:14,81K,459.68M,1.28M,...,"energy=node214,fs/disk=node214",fs/disk=0,"energy=308,fs/disk=1343606",NaN,None,05:50:41,NaN,NaN,None,NaN
10888399,default,NaN,128,1,"billing=128,cpu=128,mem=1031040M,node=1",1796,None,None,None,None,...,None,None,None,406601198.0,3057924,02:06:23,NaN,0.0,/users/3057924/msca-mip-ann-training/src,NaN
10888400,default,NaN,128,1,"cpu=128,mem=1031040M,node=1",1796,02:12:06,490K,459.68M,1.12M,...,"energy=node215,fs/disk=node215",fs/disk=0,"energy=97,fs/disk=1172450",NaN,None,02:06:23,NaN,NaN,None,NaN


In [3]:
print("Before droping empty JobID:", len(df))
print("Dropping rows with empty JobID:", len(df[df["JobID"].isna()]))
df = df[~df["JobID"].isna()]
print("After droping empty JobID:", len(df))

Before droping empty JobID: 10888402
Dropping rows with empty JobID: 0
After droping empty JobID: 10888402


In [4]:
missing_cols = df.columns[df.isnull().all()].tolist()
print("Columns with all missing values:", missing_cols)

Columns with all missing values: ['AdminComment', 'BlockID', 'Comment', 'Constraints', 'Container', 'Extra', 'Licenses', 'McsLabel', 'SegmentSize', 'SystemComment', 'WCKey', 'Unnamed: 120']


In [5]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Single-value columns:", constant_cols)

Single-value columns: ['Cluster', 'StdIn', 'Suspended', 'WCKeyID']


In [6]:
print("Before dropping:", len(df.columns))
cols_to_drop = list(set(missing_cols + constant_cols))
df = df.drop(cols_to_drop, axis=1)
print("After dropping:", len(df.columns))

Before dropping: 121
After dropping: 105


In [7]:
from pandas.util import hash_pandas_object

print("Starting.")

col_hashes = df.progress_apply(
    lambda col: hash(tuple(hash_pandas_object(col, index=False))), axis=0
)
print("Computed hash.")


duplicate_mask = col_hashes.duplicated()
duplicate_cols = df.columns[duplicate_mask].tolist()
print("Duplicates found:", duplicate_cols)
print("Finding pairs")

hash_to_col = {}
duplicate_pairs = []
for col in df.columns:
    h = col_hashes[col]
    if h in hash_to_col:
        if df[col].equals(df[hash_to_col[h]]):
            duplicate_pairs.append((hash_to_col[h], col))
    else:
        hash_to_col[h] = col


for orig, dup in duplicate_pairs:
    print(f"'{dup}' -> '{orig}'")

Starting.


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 105/105 [03:08<00:00,  1.79s/it]


Computed hash.
Duplicates found: ['NCPUS', 'ReqCPUFreqGov', 'ReqCPUFreqMax', 'ReqCPUFreqMin']
Finding pairs
'NCPUS' -> 'AllocCPUS'
'ReqCPUFreqGov' -> 'ReqCPUFreq'
'ReqCPUFreqMax' -> 'ReqCPUFreq'
'ReqCPUFreqMin' -> 'ReqCPUFreq'


In [8]:
print("Before dropping (Duplicate columns):", len(df.columns))
df = df.drop(duplicate_cols, axis=1)
print("After dropping (Duplicate columns):", len(df.columns))

Before dropping (Duplicate columns): 105
After dropping (Duplicate columns): 101


In [9]:
time_columns = [
    "AveCPU",
    "MinCPU",
    "Planned",
    "SystemCPU",
    "TotalCPU",
    "UserCPU"
]

In [10]:
def duration_to_milliseconds(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()

    # Pattern 1: D-HH:MM:SS
    match = re.match(r"(?:(\d+)-)?(\d{1,2}):(\d{2}):(\d{2})$", val)
    if match:
        days, hours, minutes, seconds = match.groups()
        total_millis = (
            (int(days) if days else 0) * 86400 * 1000
            + int(hours) * 3600 * 1000
            + int(minutes) * 60 * 1000
            + int(seconds) * 1000
        )
        return total_millis

    # Pattern 2: MM:SS.sss
    match = re.match(r"^(\d{1,2}):(\d{2}\.\d+)$", val)
    if match:
        minutes, seconds = match.groups()
        return int(minutes) * 60 * 1000 + int(float(seconds) * 1000)

    # Pattern 3: HH:MM:SS.sss
    match = re.match(r"^(\d{1,2}):(\d{2}):(\d{2}\.\d+)$", val)
    if match:
        hours, minutes, seconds = match.groups()
        return (
            int(hours) * 3600 * 1000
            + int(minutes) * 60 * 1000
            + int(float(seconds) * 1000)
        )

    print("Error:", val)
    return np.nan


for col in time_columns:
    df[col] = df[col].apply(duration_to_milliseconds).astype("Int64")

Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID
Error: INVALID


In [11]:
df[time_columns].dtypes

AveCPU       Int64
MinCPU       Int64
Planned      Int64
SystemCPU    Int64
TotalCPU     Int64
UserCPU      Int64
dtype: object

In [12]:
memory_columns = [
    "ReqMem",
    "MaxDiskWrite",
    "MaxVMSize",
    "AveRSS",
    "AveVMSize",
    "MaxRSS",
    "MaxDiskRead",
    "AveDiskRead",
    "AveDiskWrite",
]
df[memory_columns]

,ReqMem,MaxDiskWrite,MaxVMSize,AveRSS,AveVMSize,MaxRSS,MaxDiskRead,AveDiskRead,AveDiskWrite
0,800G,None,None,None,None,None,None,None,None
1,None,1351.21M,73985420K,64398820K,73985420K,64398820K,79713.93M,79713.93M,1351.21M
2,900G,None,None,None,None,None,None,None,None
3,900G,None,None,None,None,None,None,None,None
4,900G,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...
10888397,1031040M,None,None,None,None,None,None,None,None
10888398,None,1.28M,51698696K,34424564K,51698696K,34424564K,459.68M,459.68M,1.28M
10888399,1031040M,None,None,None,None,None,None,None,None
10888400,None,1.12M,18588212K,6507628K,18588212K,6507628K,459.68M,459.68M,1.12M


In [13]:
def parse_memory_to_gb(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().upper()

    match = re.fullmatch(r"([\d.]+)([KMGTP]?)", val)
    if not match:
        print("Error:", val)
        return np.nan

    number, unit = match.groups()
    number = float(number)

    unit_multipliers = {
        "": 1,  # bytes
        "K": 1024,
        "M": 1024**2,
        "G": 1024**3,
        "T": 1024**4,
        "P": 1024**5,
    }

    bytes_val = number * unit_multipliers.get(unit, 1)
    return bytes_val / 1024**3  # Convert to GB


for col in memory_columns:
    df[col] = df[col].apply(parse_memory_to_gb)

In [14]:
timestamp_columns = ["Start", "Submit", "Eligible", "End"]

for col in timestamp_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [15]:
df[timestamp_columns].dtypes

Start       datetime64[ns]
Submit      datetime64[ns]
Eligible    datetime64[ns]
End         datetime64[ns]
dtype: object

In [16]:
to_int64 = [
    "MaxDiskWriteTask",
    "MaxPagesTask",
    "MaxRSSTask",
    "MaxVMSizeTask",
    "MinCPUTask",
    "MaxDiskReadTask",
    "NNodes",
    "NTasks",
    "Priority",
    "ReqCPUS",
    "ReqNodes",
    "AllocCPUS",
    "AllocNodes",
    "AssocID",
    "CPUTimeRAW",
    "ElapsedRaw",
    "TimelimitRaw",
]

toconvert_KM = [
    "AvePages",
    "MaxPages",
    "AveCPUFreq"
]

In [17]:
df[to_int64]

,MaxDiskWriteTask,MaxPagesTask,MaxRSSTask,MaxVMSizeTask,MinCPUTask,MaxDiskReadTask,NNodes,NTasks,Priority,ReqCPUS,ReqNodes,AllocCPUS,AllocNodes,AssocID,CPUTimeRAW,ElapsedRaw,TimelimitRaw
0,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,202.0,60,1,60,2,1654,432001440,7200024,120000
1,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,NaN,1,1,1,1,1654,7200025,7200025,None
2,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,56.0,56,1,56,1,1883,30295384,540989,14399
3,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,56.0,56,1,56,1,1883,29793288,532023,14399
4,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,56.0,56,1,56,1,1883,31752560,567010,14399
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10888397,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,197.0,32,1,128,1,1796,214912,1679,360
10888398,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,NaN,128,1,128,1,1796,214912,1679,None
10888399,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,197.0,32,1,128,1,1796,935424,7308,360
10888400,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,NaN,128,1,128,1,1796,935424,7308,None


In [18]:
pattern = re.compile(r"^\d+(\.0)?$")  # Matches integers and integers ending in '.0'

for col in to_int64:
    print("Processing: ", col)
    mask = df[col].notna() & df[col].astype(str).str.match(pattern)
    df.loc[mask, col] = df.loc[mask, col].astype(float).astype("Int64")
    df.loc[~mask, col] = pd.NA

    df[col] = df[col].astype("Int64")

Processing:  MaxDiskWriteTask
Processing:  MaxPagesTask
Processing:  MaxRSSTask
Processing:  MaxVMSizeTask
Processing:  MinCPUTask
Processing:  MaxDiskReadTask
Processing:  NNodes
Processing:  NTasks
Processing:  Priority
Processing:  ReqCPUS
Processing:  ReqNodes
Processing:  AllocCPUS
Processing:  AllocNodes
Processing:  AssocID
Processing:  CPUTimeRAW
Processing:  ElapsedRaw
Processing:  TimelimitRaw


In [19]:
df["ConsumedEnergyRaw"] = pd.to_numeric(df["ConsumedEnergyRaw"], errors='coerce').astype("Float64")

In [20]:
valid_pattern = re.compile(r"^(\d+(\.\d+)?)([KkMmGg])?$")


def parse_km(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    match = valid_pattern.match(value)
    if not match:
        print(value)
        return pd.NA
    number, _, suffix = match.groups()
    number = float(number)
    if suffix in ["K", "k"]:
        number *= 1_000
    elif suffix in ["M", "m"]:
        number *= 1_000_000
    elif suffix in ["G", "g"]:
        number *= 1_000_000_000
    return int(number)


for col in toconvert_KM:
    if col in df.columns:
        print("Processing: ", col)
        df[col] = df[col].apply(parse_km).astype("Int64")

Processing:  AvePages
Processing:  MaxPages
Processing:  AveCPUFreq


In [21]:
jobid_series = df["JobID"].astype(str).str.strip()
jobid_df = pd.DataFrame(index=df.index, columns=["JobID_base", "JobID_suffix", "JobID_step", "JobID_type"])

patterns = {
    # --- Array Parent Jobs ---
    "array_range": r"^(?P<JobID_base>\d+)_(?P<JobID_suffix>\[[a-zA-Z0-9,\-%]+\])$",
    "array_job": r"^(?P<JobID_base>\d+)_(?P<JobID_suffix>\d+)$",
    
    # --- Array + Explicit Special Records (NOT steps) ---
    "array_batch": r"^(?P<JobID_base>\d+)_(?P<JobID_suffix>\d+)\.batch$",
    "array_interactive": r"^(?P<JobID_base>\d+)_(?P<JobID_suffix>\d+)\.interactive$",
    "array_special": r"^(?P<JobID_base>\d+)_(?P<JobID_suffix>\d+)\.(?P<special>[a-zA-Z]+)$", # Fallback for .extern etc.
    
    # --- Array + True Numeric Steps ---
    "array_step_num": r"^(?P<JobID_base>\d+)_(?P<JobID_suffix>\d+)\.(?P<JobID_step>\d+)$",
    
    # --- Job + Tasks ---
    "job_plus_task_step": r"^(?P<JobID_base>\d+)\+(?P<JobID_suffix>\d+)\.(?P<JobID_step>\d+)$",
    "job_plus_task": r"^(?P<JobID_base>\d+)\+(?P<JobID_suffix>\d+)$",
    
    # --- Normal Job + Explicit Special Records (NOT steps) ---
    "job_batch": r"^(?P<JobID_base>\d+)\.batch$",
    "job_interactive": r"^(?P<JobID_base>\d+)\.interactive$",
    "job_special": r"^(?P<JobID_base>\d+)\.(?P<special>[a-zA-Z]+)$", # Fallback for .extern etc.
    
    # --- Normal Job + True Numeric Steps ---
    "job_step_num": r"^(?P<JobID_base>\d+)\.(?P<JobID_step>\d+)$",
    
    # --- Base Normal Jobs ---
    "plain_job": r"^(?P<JobID_base>\d+)$"
}

ext = {k: jobid_series.str.extract(v) for k, v in patterns.items()}

mask_empty = df["JobID"].isnull() | (jobid_series.str.lower() == "none")
jobid_df.loc[mask_empty, "JobID_type"] = "Empty"

mask = ext["array_range"].notna().all(axis=1) & jobid_df["JobID_type"].isna()
if mask.any():
    jobid_df.loc[mask, ["JobID_base", "JobID_suffix"]] = ext["array_range"].loc[mask]
    
    suffixes = jobid_df.loc[mask, "JobID_suffix"]
    jobid_df.loc[mask & suffixes.str.contains('%') & suffixes.str.contains(','), "JobID_type"] = "X_[Mixed%S] (Array Mixed Stepped)"
    jobid_df.loc[mask & ~suffixes.str.contains('%') & suffixes.str.contains(','), "JobID_type"] = "X_[Mixed] (Array Mixed Range)"
    jobid_df.loc[mask & suffixes.str.contains('%') & ~suffixes.str.contains(','), "JobID_type"] = "X_[Y-Z%S] (Array Range Stepped)"
    jobid_df.loc[mask & ~suffixes.str.contains('%') & ~suffixes.str.contains(',') & suffixes.str.contains('-'), "JobID_type"] = "X_[Y-Z] (Array Range)"
    jobid_df.loc[mask & jobid_df["JobID_type"].isna(), "JobID_type"] = "X_[Y] (Array Single Element)"

standard_mappings = [
    ("array_batch", "X_Y.batch (Array Batch)"),
    ("array_interactive", "X_Y.interactive (Array Interactive)"),
    ("job_batch", "X.batch (Normal Batch)"),
    ("job_interactive", "X.interactive (Normal Interactive)"),
    ("array_step_num", "X_Y.Z (Array Step)"),
    ("job_plus_task_step", "X+Y.Z (Job Task Step)"),
    ("job_step_num", "X.Y (Job Step)"),
    ("array_job", "X_Y (Array Job)"),
    ("job_plus_task", "X+Y (Job plus Task)"),
    ("plain_job", "X (Normal)")
]

for key, label in standard_mappings:
    mask = ext[key].notna().all(axis=1) & jobid_df["JobID_type"].isna()
    if mask.any():
        available_cols = ext[key].columns.intersection(jobid_df.columns)
        jobid_df.loc[mask, available_cols] = ext[key].loc[mask, available_cols]
        jobid_df.loc[mask, "JobID_type"] = label

mask = ext["job_special"].notna().all(axis=1) & jobid_df["JobID_type"].isna()
if mask.any():
    jobid_df.loc[mask, "JobID_base"] = ext["job_special"].loc[mask, "JobID_base"]
    jobid_df.loc[mask, "JobID_type"] = "X." + ext["job_special"].loc[mask, "special"] + " (Normal " + ext["job_special"].loc[mask, "special"] + ")"

mask = ext["array_special"].notna().all(axis=1) & jobid_df["JobID_type"].isna()
if mask.any():
    jobid_df.loc[mask, ["JobID_base", "JobID_suffix"]] = ext["array_special"].loc[mask, ["JobID_base", "JobID_suffix"]]
    jobid_df.loc[mask, "JobID_type"] = "X_Y." + ext["array_special"].loc[mask, "special"] + " (Array " + ext["array_special"].loc[mask, "special"] + ")"

unknown_mask = jobid_df["JobID_type"].isna()
jobid_df.loc[unknown_mask, "JobID_base"] = df.loc[unknown_mask, "JobID"]
jobid_df.loc[unknown_mask, "JobID_type"] = "unknown"

for col in ["JobID_base", "JobID_suffix", "JobID_step", "JobID_type"]:
    jobid_df[col] = jobid_df[col].where(pd.notnull(jobid_df[col]), None)

jobid_df = jobid_df[["JobID_base", "JobID_suffix", "JobID_step", "JobID_type"]]

In [22]:
df = pd.concat([df, jobid_df], axis=1)
print(len(df))

10888402


In [23]:
valid_partitions = df.dropna(subset=['Partition'])
partition_variance = valid_partitions.groupby('JobID_base')['Partition'].nunique()
discrepancies = partition_variance[partition_variance > 1]

if discrepancies.empty:
    print("Success: No discrepancies found. Every JobID_base is associated with exactly one partition.")
else:
    print(f"Discrepancy Alert: Found {len(discrepancies)} JobID_base(s) associated with multiple partitions.")
    problem_ids = discrepancies.index.tolist()
    print(valid_partitions[valid_partitions['JobID_base'].isin(problem_ids)][['JobID_base', 'Partition']].drop_duplicates().sort_values('JobID_base'))

Discrepancy Alert: Found 250 JobID_base(s) associated with multiple partitions.
         JobID_base                           Partition
492911      3086475                           k2-medpri
492921      3086475                            k2-himem
492931      3086485                           k2-medpri
492939      3086485                            k2-himem
494108      3086970                           k2-medpri
...             ...                                 ...
10872279    7595178  k2-math-physics,k2-hipri,k2-medpri
10872280    7595322                     k2-math-physics
10872298    7595322                            k2-hipri
10873079    7595728                     k2-math-physics
10873413    7595728                            k2-hipri

[544 rows x 2 columns]


In [24]:
partition_counts = df.dropna(subset=['Partition']).groupby('JobID_base')['Partition'].nunique()
mixed_job_ids = partition_counts[partition_counts > 1].index

def get_partition_label(group):
    unique_parts = group.unique()
    if len(unique_parts) > 1:
        return "MIXED"
    return unique_parts[0]

partition_lookup = df.dropna(subset=['Partition']).groupby('JobID_base')['Partition'].apply(get_partition_label)
df['Partition'] = df['JobID_base'].map(partition_lookup)

In [25]:
df.to_parquet("data/slurm-preprocessed.parquet", index=False)

In [26]:
df["JobID_type"].value_counts().sum()

np.int64(10888402)

In [27]:
custom_order = [
    'k2-lowpri', 'k2-gpu', 'k2-living-labs', 'k2-gpu-intel', 'k2-gpu-a100mig', 
    'k2-bioinf', 'k2-medpri', 'k2-gpu-a100', 'k2-epsrc-gpu-v100', 'k2-gpu-v100', 
    'k2-epsrc', 'k2-epsrc-gpu-a100', 'k2-epsrc-gpu-h100', 'k2-gpu-h100', 
    'k2-sandbox', 'k2-himem', 'k2-gpu-amd', 'k2-hipri', 'k2-gpu-interactive', 
    'k2-epsrc-gpu', 'k2-math-physics-debug', 'k2-fmipr', 'k2-math-physics', 
    'k2-gen17', 'k2-amd-xseries', 'k2-epsrc-himem', 'k2-epsrc-gpu-amd', 
    'k2-gpu-exotic', 'k2-hysafer', 'MIXED'
]

type_counts = df[['Partition', 'JobID_type']].value_counts(
    ['Partition', 'JobID_type'], 
    dropna=False
).reset_index(name='Count')
type_counts['Partition'] = pd.Categorical(
    type_counts['Partition'], 
    categories=custom_order, 
    ordered=True
)
type_counts = type_counts.sort_values(by=['Partition', 'Count'], ascending=[True, False])
type_counts

,Partition,JobID_type,Count
16,k2-lowpri,X (Normal),63219
24,k2-lowpri,X_Y (Array Job),28970
25,k2-lowpri,X.batch (Normal Batch),25973
28,k2-lowpri,X_Y.batch (Array Batch),22832
86,k2-lowpri,X.Y (Job Step),1259
...,...,...,...
288,NaN,X_[Y-Z] (Array Range),1
289,NaN,X_[Y-Z] (Array Range),1
290,NaN,X (Normal),1
291,NaN,X (Normal),1
